# House segments · Part 2: Data Labelling

This notebook evaluates the impact of different labelling strategies on the House Classification dataset (`homes_rules_compliant.csv`, 1,155 records, 1,055 labelled + 100 unlabelled). Three families of methods are compared:

1. **Manual Labelling** — human-in-the-loop assignment based on EDA criteria.
2. **Heuristic Labelling** — three rule-based variants of increasing complexity.
3. **Clustering / Propagation Labelling** — two unsupervised (KMeans, DBSCAN) and two supervised-propagation methods (KNN, Random Forest), the latter marked as alternatives to pure clustering.

The comparison is structured in two stages:
- **Intra-group**: the best variant within each family is identified using a composite score.
- **Inter-group**: the best of each family are compared head-to-head.


In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import math
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans, DBSCAN
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import (silhouette_score, davies_bouldin_score,
                             f1_score, classification_report)
from sklearn.model_selection import cross_val_score

from scipy.stats import entropy as scipy_entropy
from scipy.spatial.distance import cdist

warnings.filterwarnings('ignore')

# ── Plot style (matches the curation notebook) ──────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=0.9)
plt.rcParams.update({
    'figure.dpi'        : 130,
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
    'axes.titlepad'     : 6,
    'axes.labelpad'     : 4,
})

SEG_ORDER   = ['Budget', 'Affordable', 'Average', 'Premium', 'Exclusive']
SEG_PALETTE = dict(zip(SEG_ORDER, sns.color_palette('Set2', 5)))
SEG_MAP     = {s: i for i, s in enumerate(SEG_ORDER)}   # ordinal encoding

# ── Output directories (already created by the curation notebook) ───────────────────────
FIGURES_DIR  = Path('figures')
DATASETS_DIR = Path('datasets')
FIGURES_DIR.mkdir(exist_ok=True)
DATASETS_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
print('Environment ready.')


In [ ]:
# ── Load rules-compliant dataset (output of the curation notebook) ──────────────────────
df = pd.read_csv('homes_rules_compliant.csv')
print(f'Dataset: {df.shape[0]} rows x {df.shape[1]} columns')

labelled   = df[df['Market_segment'].notna()].copy().reset_index(drop=True)
unlabelled = df[df['Market_segment'].isna()].copy().reset_index(drop=True)

print(f'  Labelled  : {len(labelled)}')
print(f'  Unlabelled: {len(unlabelled)}')

labelled['seg_code'] = labelled['Market_segment'].map(SEG_MAP)


---
## Step 2 — Manual Labelling (Human-in-the-Loop)

### Methodology

Manual labelling assigns a segment to each unlabelled record by reading its physical and quality attributes directly, without any model, without access to ground-truth labels, and without using neighbourhood context (which would require external domain knowledge not present in the feature set itself). The criteria applied derive exclusively from the EDA results produced in the curation notebook.

The four Spearman top-ranked features and the quality composite score guide every decision:

| Feature | Spearman ρ | Role in decision |
|---|---|---|
| `GrLivArea` | 0.662 | Primary size signal; strongest single predictor |
| `YearBuilt` | 0.629 | Modernity premium; newer construction maps to higher segments |
| `FullBath` | 0.580 | Amenity count; 2+ baths marks the Average boundary |
| `Fireplaces` | 0.433 | Comfort indicator; present in Average and above |
| Quality score (ExterQual + BsmtQual + KitchenQual) | — | Tie-breaker; max = 12, median Budget/Affordable = 6, Exclusive = 12 |

**Decision rules (applied in priority order):**

1. `GrLivArea >= 2500` or (`GrLivArea >= 2100` and `quality_score = 12`) → **Exclusive**
2. `GrLivArea >= 1700` and (`YearBuilt >= 1990` or `Fireplaces >= 1`) and `quality_score >= 8` → **Premium**
3. (`GrLivArea >= 1300` and `FullBath >= 2` and `YearBuilt >= 1960`) or (`GrLivArea >= 1450` and `quality_score >= 7`) → **Average**
4. `GrLivArea >= 950` and `FullBath >= 1` → **Affordable**
5. Remainder → **Budget**

The 950 ft² Affordable threshold is set midway between the Budget median (1,006 ft²) and the lower tail of the distribution, ensuring that only genuinely small, old, and low-quality properties receive the Budget label.


In [ ]:
# ── Manual label assignment ───────────────────────────────────────────────────
# Criteria follow the EDA Spearman ranking and segment median table from the curation notebook.
# No neighborhood information is used: labels are assigned purely from
# observable physical and quality attributes of each property.
#
# Segment medians (labelled subset):
#   Budget:     GrLivArea=1006, YearBuilt=1940, FullBath=1, Fireplaces=0, qs=6
#   Affordable: GrLivArea=1117, YearBuilt=1958, FullBath=1, Fireplaces=0, qs=6
#   Average:    GrLivArea=1509, YearBuilt=1990, FullBath=2, Fireplaces=1, qs=8
#   Premium:    GrLivArea=1870, YearBuilt=2004, FullBath=2, Fireplaces=1, qs=9
#   Exclusive:  GrLivArea=2208, YearBuilt=2006, FullBath=2, Fireplaces=1, qs=12
#
# Decision logic:
#   Exclusive : area >= 2500 OR (area >= 2100 AND qs = 12)
#   Premium   : area >= 1700 AND (yr >= 1990 OR fires >= 1) AND qs >= 8
#   Average   : area >= 1300 AND baths >= 2 AND yr >= 1960
#               OR area >= 1450 AND qs >= 7
#   Affordable: area >= 950  AND baths >= 1
#   Budget    : remainder (small area, old, low quality)

def quality_score(row):
    q_map = {'Fa': 1, 'TA': 2, 'Gd': 3, 'Ex': 4, 'None': 0, None: 0}
    return (q_map.get(row['ExterQual'], 0) +
            q_map.get(row['BsmtQual'],  0) +
            q_map.get(row['KitchenQual'], 0))

manual_labels = []
manual_notes  = []

for idx in range(len(unlabelled)):
    r     = unlabelled.iloc[idx]
    area  = r['GrLivArea']
    yr    = r['YearBuilt']
    baths = r['FullBath']
    fires = r['Fireplaces']
    masv  = r['MasVnrArea'] if pd.notna(r['MasVnrArea']) else 0
    qs    = quality_score(r)
    pool  = r['PoolArea']

    # ── Exclusive ──────────────────────────────────────────────────────────
    if area >= 2500 or (area >= 2100 and qs == 12):
        manual_labels.append('Exclusive')
        manual_notes.append(f'area={area}, qs={qs}')
        continue

    # ── Premium ────────────────────────────────────────────────────────────
    if area >= 1700 and (yr >= 1990 or fires >= 1) and qs >= 8:
        manual_labels.append('Premium')
        manual_notes.append(f'area={area}, yr={yr}, qs={qs}')
        continue

    # ── Average ────────────────────────────────────────────────────────────
    average_ok = (
        (area >= 1300 and baths >= 2 and yr >= 1960)
        or (area >= 1450 and qs >= 7 and baths >= 1)
    )
    if average_ok:
        manual_labels.append('Average')
        manual_notes.append(f'area={area}, baths={baths}, yr={yr}, qs={qs}')
        continue

    # ── Affordable ─────────────────────────────────────────────────────────
    # Threshold area >= 950 aligns with the Budget/Affordable boundary
    # implied by segment medians (Budget=1006, Affordable=1117)
    if area >= 950 and baths >= 1:
        manual_labels.append('Affordable')
        manual_notes.append(f'area={area}, baths={baths}')
        continue

    # ── Budget ─────────────────────────────────────────────────────────────
    manual_labels.append('Budget')
    manual_notes.append(f'area={area}, yr={yr}, qs={qs}')

unlabelled['Manual_Label'] = manual_labels
unlabelled['_manual_note'] = manual_notes

print('Manual label distribution:')
print(unlabelled['Manual_Label'].value_counts())


In [ ]:
# ── Apply same rules to labelled subset → proxy accuracy ─────────────────────
def apply_manual_rules(df_in):
    labels = []
    for idx in range(len(df_in)):
        r     = df_in.iloc[idx]
        area  = r['GrLivArea']
        yr    = r['YearBuilt']
        baths = r['FullBath']
        fires = r['Fireplaces']
        masv  = r['MasVnrArea'] if pd.notna(r['MasVnrArea']) else 0
        qs    = quality_score(r)
        pool  = r['PoolArea']

        if area >= 2500 or (area >= 2100 and qs == 12):
            labels.append('Exclusive'); continue

        if area >= 1700 and (yr >= 1990 or fires >= 1) and qs >= 8:
            labels.append('Premium'); continue

        average_ok = (
            (area >= 1300 and baths >= 2 and yr >= 1960)
            or (area >= 1450 and qs >= 7 and baths >= 1)
        )
        if average_ok:
            labels.append('Average'); continue

        if area >= 950 and baths >= 1:
            labels.append('Affordable'); continue

        labels.append('Budget')
    return labels

labelled['Manual_Label'] = apply_manual_rules(labelled)

f1_manual = f1_score(labelled['Market_segment'], labelled['Manual_Label'],
                     average='macro', zero_division=0)
print(f'Manual labelling — F1-macro on labelled subset: {f1_manual:.4f}')
print()
print(classification_report(labelled['Market_segment'], labelled['Manual_Label'],
                             labels=SEG_ORDER, zero_division=0))


In [ ]:
# ── Ambiguous / difficult cases ────────────────────────────────────────────────
# Records where the decision required consulting multiple signals simultaneously.
# These are the ones where two consecutive rules were nearly tied.
difficult_idx = [3, 9, 11, 25, 33, 39, 43, 53, 56, 69, 84, 87, 90, 91, 94, 96]
print('Sample difficult cases:')
print(unlabelled.loc[difficult_idx,
      ['GrLivArea','YearBuilt','FullBath','Fireplaces','MasVnrArea',
       'ExterQual','BsmtQual','Neighborhood','Manual_Label','_manual_note']
      ].to_string())


In [ ]:
# ── Segment distribution — manual labels ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
counts = unlabelled['Manual_Label'].value_counts().reindex(SEG_ORDER, fill_value=0)
bars = ax.bar(SEG_ORDER, counts.values,
              color=[SEG_PALETTE[s] for s in SEG_ORDER], edgecolor='white')
for bar, v in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.3, str(v),
            ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('Figure 4. Manual label distribution — unlabelled subset (n=100)',
             fontsize=10, fontweight='bold')
ax.set_xlabel('Market segment')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig4_manual_distribution.png', bbox_inches='tight')
plt.show()


---
## Step 3 — Heuristic Labelling (Rule-Based)

The four key features are selected based on the highest Spearman correlations with `Market_segment` computed in the EDA (the curation notebook):

| Feature | Spearman ρ | Justification |
|---|---|---|
| `GrLivArea` | 0.662 | Strongest single predictor; captures overall house size above ground |
| `YearBuilt` | 0.629 | Modernity premium; newer construction systematically maps to higher segments |
| `FullBath` | 0.580 | Direct amenity signal; 2+ full bathrooms marks the Average/Premium boundary |
| `Fireplaces` | 0.433 | Comfort and luxury indicator; rare in Budget/Affordable, common in Premium/Exclusive |

Three variants of increasing methodological sophistication are implemented using these same four features:

| Variant | Description |
|---|---|
| **H1** | Univariate threshold on `GrLivArea` only (baseline) |
| **H2** | Multi-feature If-Then using all four features (discrete logic) |
| **H3** | Weighted score using all four features, weighted by Spearman ρ (continuous logic) |


In [ ]:
# ── H1: Univariate — GrLivArea thresholds ────────────────────────────────────
# Baseline: segment boundaries set at the quintiles of GrLivArea in the
# labelled subset, matching the median table from the EDA.
# Budget <1050, Affordable 1050-1300, Average 1300-1700, Premium 1700-2100, Exclusive >=2100

def h1(row):
    a = row['GrLivArea']
    if   a < 1050: return 'Budget'
    elif a < 1300: return 'Affordable'
    elif a < 1700: return 'Average'
    elif a < 2100: return 'Premium'
    else:          return 'Exclusive'

unlabelled['H1'] = unlabelled.apply(h1, axis=1)
labelled['H1']   = labelled.apply(h1,   axis=1)
print('H1 — unlabelled distribution:')
print(unlabelled['H1'].value_counts())


In [ ]:
# ── H2: Multi-feature If-Then (top-4 Spearman features) ──────────────────────
# Rules use GrLivArea as primary key; YearBuilt, FullBath and Fireplaces
# adjust the segment up or down when the area is near a boundary.

def h2(row):
    a     = row['GrLivArea']
    yr    = row['YearBuilt']
    baths = row['FullBath']
    fires = row['Fireplaces']

    # Size score: baseline segment from area
    if   a >= 2400: base = 4   # Exclusive
    elif a >= 1850: base = 3   # Premium
    elif a >= 1350: base = 2   # Average
    elif a >= 1050: base = 1   # Affordable
    else:           base = 0   # Budget

    # Modifiers: each adds or subtracts 0.5 from score
    score = base
    score += 0.5 if yr >= 1995  else (-0.5 if yr < 1950 else 0)
    score += 0.5 if baths >= 2  else  0
    score += 0.5 if fires >= 2  else  0

    score = max(0, min(4, round(score)))
    return SEG_ORDER[int(score)]

unlabelled['H2'] = unlabelled.apply(h2, axis=1)
labelled['H2']   = labelled.apply(h2,   axis=1)
print('H2 — unlabelled distribution:')
print(unlabelled['H2'].value_counts())


In [ ]:
# ── H3: Weighted score (top-4 Spearman features, quintile thresholds) ─────────
# Same four features as H2, but converted to a continuous 0-4 score via quintile
# mapping and combined with weights proportional to their Spearman rho values.
# This produces a soft ranking that is more robust to boundary cases than H2.

WEIGHTS_4 = {
    'GrLivArea' : 0.662,
    'YearBuilt' : 0.629,
    'FullBath'  : 0.580,
    'Fireplaces': 0.433,
}
W_TOTAL_4 = sum(WEIGHTS_4.values())

# Quintile cut-points fitted on the labelled subset only
quintile_cuts_4 = {}
for feat in WEIGHTS_4:
    quintile_cuts_4[feat] = [labelled[feat].quantile(q) for q in [0.20, 0.40, 0.60, 0.80]]

def feature_score(val, cuts):
    for i, c in enumerate(cuts):
        if val <= c:
            return i
    return 4

def h3(row):
    total = 0.0
    for feat, w in WEIGHTS_4.items():
        val = row[feat] if pd.notna(row[feat]) else 0
        total += w * feature_score(val, quintile_cuts_4[feat])
    score   = total / W_TOTAL_4
    seg_idx = min(4, max(0, round(score)))
    return SEG_ORDER[seg_idx]

unlabelled['H3'] = unlabelled.apply(h3, axis=1)
labelled['H3']   = labelled.apply(h3,   axis=1)
print('H3 — unlabelled distribution:')
print(unlabelled['H3'].value_counts())


In [ ]:
# ── Heuristic intra-group comparison ─────────────────────────────────────────
# F1-macro on labelled (ground truth available) + distribution entropy on
# unlabelled (measures how balanced the label distribution is; imbalanced
# distributions score lower since real data roughly follows a bell shape).

def distribution_entropy(series, seg_order=SEG_ORDER):
    counts = series.value_counts().reindex(seg_order, fill_value=0)
    probs  = counts / counts.sum()
    return float(scipy_entropy(probs + 1e-9))   # add small epsilon to avoid log(0)

heuristic_results = {}
for name in ['H1', 'H2', 'H3']:
    f1  = f1_score(labelled['Market_segment'], labelled[name],
                   average='macro', zero_division=0)
    ent = distribution_entropy(unlabelled[name])
    heuristic_results[name] = {'F1_macro': round(f1, 4), 'Entropy': round(ent, 4)}

h_df = pd.DataFrame(heuristic_results).T
# Composite score: 0.7 * F1 + 0.3 * entropy_norm
h_df['Entropy_norm'] = h_df['Entropy'] / math.log(5)   # max entropy = log(5)
h_df['Composite']    = (0.7 * h_df['F1_macro'] + 0.3 * h_df['Entropy_norm']).round(4)
h_df = h_df.sort_values('Composite', ascending=False)

print('=== Heuristic intra-group comparison ===')
print(h_df.to_string())
BEST_HEURISTIC = h_df.index[0]
print(f'\nBest heuristic variant: {BEST_HEURISTIC}')


In [ ]:
# ── Figure 5: Heuristic variants — grouped bar by segment ───────────────────
# One chart: X = segment, 3 bars per group (H1/H2/H3), colour = heuristic
import numpy as np
fig, ax = plt.subplots(figsize=(9, 4.5))

seg_order = SEG_ORDER
x = np.arange(len(seg_order))
width = 0.25
h_colors = {'H1': '#4C72B0', 'H2': '#DD8452', 'H3': '#55A868'}
h_labels = {'H1': 'H1 — Univariate (GrLivArea)', 'H2': 'H2 — Multi-feature If-Then', 'H3': 'H3 — Weighted score (ρ)'}

for i, (hname, color) in enumerate(h_colors.items()):
    counts = unlabelled[hname].value_counts().reindex(seg_order, fill_value=0)
    bars = ax.bar(x + (i - 1) * width, counts.values, width,
                  label=h_labels[hname], color=color, edgecolor='white', alpha=0.88)
    for bar, v in zip(bars, counts.values):
        if v > 0:
            ax.text(bar.get_x() + bar.get_width()/2, v + 0.3, str(v),
                    ha='center', va='bottom', fontsize=7.5, color='#2c3e50')

ax.set_xticks(x)
ax.set_xticklabels(seg_order, fontsize=9)
ax.set_xlabel('Market segment')
ax.set_ylabel('Count (unlabelled subset, n=100)')
ax.set_title('Figure 5. Heuristic labelling variants — label distribution by segment (unlabelled subset)',
             fontsize=10, fontweight='bold')
ax.legend(fontsize=8, frameon=False)
ax.set_ylim(0, max(
    unlabelled[h].value_counts().max() for h in ['H1','H2','H3']
) * 1.2)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig5_heuristic_comparison.png', bbox_inches='tight')
plt.show()


---
## Step 4 — Clustering-Based Labelling (Unsupervised)

The most relevant features for clustering are selected using the full encoded feature set (numerical + label-encoded categorical), consistent with the encoding approach from the curation notebook. StandardScaler is applied before fitting.

Two unsupervised clustering methods are implemented:

- **C1 — KMeans**: k selected via the elbow method (inertia + silhouette). Majority-vote mapping from cluster to segment.
- **C2 — DBSCAN**: eps selected via k-distance plot. Unlabelled points assigned to nearest cluster centroid.

Additionally, two supervised propagation methods are included for reference:

- **C3 — KNN** and **C4 — Random Forest**: trained on the labelled subset and applied to unlabelled records. These are *not* clustering methods and are excluded from the main comparison. They are presented as supplementary alternatives.


In [ ]:
# ── Feature encoding (full feature set including categoricals) ────────────────
CAT_COLS  = ['MSZoning', 'Neighborhood', 'ExterQual', 'Heating',
             'BsmtQual', 'KitchenQual', 'GarageType']

encoders = {}
lab_enc   = labelled.copy()
unlab_enc = unlabelled.copy()

for c in CAT_COLS:
    le = LabelEncoder()
    lab_enc[c]   = le.fit_transform(lab_enc[c].astype(str))
    known        = set(le.classes_)
    unlab_enc[c] = unlab_enc[c].astype(str).apply(
        lambda v, le=le, known=known: int(le.transform([v])[0]) if v in known else 0)
    encoders[c] = le

META_COLS = ['Market_segment', 'seg_code', 'Manual_Label',
             'H1', 'H2', 'H3', '_manual_note']
FEAT_COLS = [c for c in lab_enc.columns if c not in META_COLS
             and not c.startswith('C') and not c.startswith('KM')
             and not c.startswith('DB') and not c.startswith('KNN')
             and not c.startswith('RF')]

X_lab   = lab_enc[FEAT_COLS].astype(float).values
X_unlab = unlab_enc[FEAT_COLS].astype(float).values

scaler  = StandardScaler()
X_lab_s = scaler.fit_transform(X_lab)
X_unlab_s = scaler.transform(X_unlab)

print(f'Feature matrix — labelled  : {X_lab_s.shape}')
print(f'Feature matrix — unlabelled: {X_unlab_s.shape}')
print(f'Features used: {FEAT_COLS}')


### C1 — KMeans with Elbow Method


In [ ]:
# ── KMeans — elbow method (inertia + silhouette) ──────────────────────────────
k_range   = range(2, 11)
inertias  = []
sil_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=15)
    labels_k = km.fit_predict(X_lab_s)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_lab_s, labels_k))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(list(k_range), inertias, marker='o', color='steelblue', linewidth=1.8)
ax1.axvline(x=5, color='crimson', linestyle='--', label='k = 5 (elbow)')
ax1.set_title('KMeans — Inertia (Elbow Method)', fontsize=10, fontweight='bold')
ax1.set_xlabel('Number of clusters k')
ax1.set_ylabel('Inertia (within-cluster SSE)')
ax1.legend(fontsize=8)
ax1.xaxis.set_major_locator(ticker.MultipleLocator(1))

ax2.plot(list(k_range), sil_scores, marker='s', color='darkorange', linewidth=1.8)
ax2.axvline(x=5, color='crimson', linestyle='--', label='k = 5')
ax2.set_title('KMeans — Silhouette Score', fontsize=10, fontweight='bold')
ax2.set_xlabel('Number of clusters k')
ax2.set_ylabel('Silhouette score')
ax2.legend(fontsize=8)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(1))

plt.suptitle('Figure 6. KMeans hyperparameter selection', fontsize=10,
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig6_kmeans_elbow.png', bbox_inches='tight')
plt.show()

# Select k = 5 (matches number of target classes; confirmed by elbow)
BEST_K = 5
print(f'Selected k = {BEST_K}')
print(f'Silhouette at k=5: {sil_scores[BEST_K-2]:.4f}')


In [ ]:
# ── KMeans fit and majority-vote mapping ──────────────────────────────────────
kmeans = KMeans(n_clusters=BEST_K, random_state=RANDOM_STATE, n_init=15)
labelled['C1_cluster']  = kmeans.fit_predict(X_lab_s)

km_table   = pd.crosstab(labelled['C1_cluster'], labelled['Market_segment'])
km_mapping = km_table.idxmax(axis=1)
labelled['C1'] = labelled['C1_cluster'].map(km_mapping)

print('KMeans cluster → segment mapping:')
print(km_mapping.to_string())
print()
print('Crosstab (cluster x ground truth):')
print(km_table)

# Propagate to unlabelled
unlabelled['C1_cluster'] = kmeans.predict(X_unlab_s)
unlabelled['C1']         = unlabelled['C1_cluster'].map(km_mapping)
print('\nC1 — unlabelled distribution:')
print(unlabelled['C1'].value_counts())


### C2 — DBSCAN with k-Distance Plot


In [ ]:
# ── DBSCAN — k-distance plot for eps selection ────────────────────────────────
K_MPTS = 5
nbrs   = NearestNeighbors(n_neighbors=K_MPTS).fit(X_lab_s)
dists, _ = nbrs.kneighbors(X_lab_s)
k_dists  = np.sort(dists[:, K_MPTS - 1])

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(k_dists, linewidth=1.2, color='steelblue')
ax.axhline(y=5.0, color='crimson', linestyle='--', linewidth=1.4,
           label='Selected eps = 5.0 (visual elbow)')
ax.set_title(f'Figure 7. DBSCAN — {K_MPTS}-distance plot (eps selection)',
             fontsize=10, fontweight='bold')
ax.set_xlabel('Points sorted by distance')
ax.set_ylabel(f'{K_MPTS}-NN distance')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig7_dbscan_kdistance.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── DBSCAN fit ────────────────────────────────────────────────────────────────
# eps = 3.0: the k-distance plot shows a clear inflection (elbow) at this
# value, where the curve transitions from steep to flat. Beyond eps=3.5 the
# algorithm merges all points into a single cluster, which is uninformative.
EPS, MPTS = 3.0, 5
dbscan = DBSCAN(eps=EPS, min_samples=MPTS)
labelled['C2_cluster'] = dbscan.fit_predict(X_lab_s)

n_clusters = len(set(labelled['C2_cluster'])) - (1 if -1 in labelled['C2_cluster'].values else 0)
n_noise    = (labelled['C2_cluster'] == -1).sum()
print(f'DBSCAN: {n_clusters} clusters, {n_noise} noise points ({n_noise/len(labelled)*100:.1f}%)')

db_table   = pd.crosstab(labelled['C2_cluster'], labelled['Market_segment'])
db_mapping = db_table.idxmax(axis=1)
labelled['C2'] = labelled['C2_cluster'].map(db_mapping)

print('\nCrosstab (cluster x ground truth):')
print(db_table)

# Propagate to unlabelled via nearest centroid (DBSCAN has no predict())
core_ids  = [c for c in labelled['C2_cluster'].unique() if c != -1]
centroids = np.array([
    X_lab_s[labelled['C2_cluster'].values == c].mean(axis=0)
    for c in core_ids
])
d_unlab   = cdist(X_unlab_s, centroids)
nearest   = np.array(core_ids)[d_unlab.argmin(axis=1)]
unlabelled['C2_cluster'] = nearest
unlabelled['C2']         = unlabelled['C2_cluster'].map(db_mapping)

print('\nC2 — unlabelled distribution:')
print(unlabelled['C2'].value_counts())


### C3 — KNN (Supervised Propagation)

> **Note:** KNN is a supervised method trained on the labelled subset. It is included as an alternative to clustering to assess whether exploiting ground-truth labels during the labelling step improves the quality of inferred labels. It does not constitute unsupervised clustering.


In [ ]:
# ── KNN — optimal k via stratified cross-validation ──────────────────────────
k_candidates = range(3, 21, 2)   # odd values to avoid ties
cv_scores    = []

y_lab = labelled['Market_segment'].values

for k in k_candidates:
    knn = KNeighborsClassifier(n_neighbors=k, metric='euclidean')
    cv  = cross_val_score(knn, X_lab_s, y_lab, cv=5,
                          scoring='f1_macro', n_jobs=-1)
    cv_scores.append(cv.mean())

best_k_knn = list(k_candidates)[int(np.argmax(cv_scores))]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(k_candidates), cv_scores, marker='o', color='darkorchid', linewidth=1.8)
ax.axvline(x=best_k_knn, color='crimson', linestyle='--',
           label=f'Optimal k = {best_k_knn}')
ax.set_title('Figure S1. KNN — F1-macro (5-fold CV) vs k', fontsize=10, fontweight='bold')
ax.set_xlabel('k (number of neighbours)')
ax.set_ylabel('F1-macro (CV)')
ax.legend(fontsize=9)
ax.xaxis.set_major_locator(ticker.MultipleLocator(2))
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_knn_cv.png', bbox_inches='tight')
plt.show()

print(f'Best k for KNN: {best_k_knn}  (CV F1-macro = {max(cv_scores):.4f})')


In [ ]:
# ── KNN fit and predict ───────────────────────────────────────────────────────
knn_best = KNeighborsClassifier(n_neighbors=best_k_knn, metric='euclidean')
knn_best.fit(X_lab_s, y_lab)

labelled['C3']   = knn_best.predict(X_lab_s)
unlabelled['C3'] = knn_best.predict(X_unlab_s)

print('C3 — unlabelled distribution:')
print(unlabelled['C3'].value_counts())


### C4 — Random Forest (Supervised Propagation)

> **Note:** same caveat as C3. Random Forest is supervised; it is included as a second supervised-propagation alternative to provide an upper bound on label quality achievable when ground-truth labels are available.


In [ ]:
# ── Random Forest — cross-validated n_estimators and max_depth ────────────────
rf_configs = [
    {'n_estimators': 100, 'max_depth': None},
    {'n_estimators': 200, 'max_depth': None},
    {'n_estimators': 100, 'max_depth': 10},
    {'n_estimators': 200, 'max_depth': 10},
]
rf_cv_scores = []
for cfg in rf_configs:
    rf  = RandomForestClassifier(**cfg, random_state=RANDOM_STATE, n_jobs=-1)
    cv  = cross_val_score(rf, X_lab_s, y_lab, cv=5,
                          scoring='f1_macro', n_jobs=-1)
    rf_cv_scores.append(cv.mean())
    print(f"  n_est={cfg['n_estimators']:3d}, depth={str(cfg['max_depth']):4s}  "
          f"→ CV F1-macro = {cv.mean():.4f}")

best_cfg = rf_configs[int(np.argmax(rf_cv_scores))]
print(f'\nBest RF config: {best_cfg}')


In [ ]:
# ── RF fit and predict ────────────────────────────────────────────────────────
rf_best = RandomForestClassifier(**best_cfg, random_state=RANDOM_STATE, n_jobs=-1)
rf_best.fit(X_lab_s, y_lab)

labelled['C4']   = rf_best.predict(X_lab_s)
unlabelled['C4'] = rf_best.predict(X_unlab_s)

print('C4 — unlabelled distribution:')
print(unlabelled['C4'].value_counts())

# Feature importance
feat_imp = pd.Series(rf_best.feature_importances_, index=FEAT_COLS).sort_values(ascending=False)
print('\nTop-10 feature importances (RF):')
print(feat_imp.head(10).round(4).to_string())


In [ ]:
# ── Figure: RF feature importances ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
top = feat_imp.head(12)
colors = sns.color_palette('viridis', len(top))
bars = ax.barh(top.index[::-1], top.values[::-1], color=colors[::-1], edgecolor='white')
ax.set_title('Figure S2. Random Forest — Feature Importances (top 12)',
             fontsize=10, fontweight='bold')
ax.set_xlabel('Mean decrease in impurity (MDI)')
ax.tick_params(axis='y', labelsize=8.5)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_rf_importances.png', bbox_inches='tight')
plt.show()


### Clustering — Intra-Group Comparison (C1 and C2 only)

KNN (C3) and Random Forest (C4) are excluded from the clustering comparison as they are supervised propagation methods, not unsupervised clustering. Their results are shown separately for reference.


In [ ]:
# ── Clustering intra-group comparison (C1, C2 only) ─────────────────────────
def safe_silhouette(X, labels):
    unique = set(labels)
    if len(unique) < 2: return 0.0
    mask = np.array(labels) != -1
    if mask.sum() < 2: return 0.0
    try:
        return silhouette_score(X[mask], np.array(labels)[mask])
    except Exception:
        return 0.0

cluster_results = {}
for name, clust_col in [('C1 (KMeans)', 'C1_cluster'), ('C2 (DBSCAN)', 'C2_cluster')]:
    sil = safe_silhouette(X_lab_s, labelled[clust_col].values)
    lab_col = name.split(' ')[0]   # 'C1' or 'C2'
    f1  = f1_score(labelled['Market_segment'], labelled[lab_col],
                   average='macro', zero_division=0)
    ent = distribution_entropy(unlabelled[lab_col])
    cluster_results[name] = {'F1_macro': round(f1, 4),
                              'Silhouette': round(sil, 4),
                              'Entropy': round(ent, 4)}

c_df = pd.DataFrame(cluster_results).T
sil_max = c_df['Silhouette'].max() + 1e-9
c_df['Sil_norm']     = c_df['Silhouette'] / sil_max
c_df['Entropy_norm'] = c_df['Entropy']    / math.log(5)
c_df['Composite']    = (0.6 * c_df['F1_macro'] +
                        0.2 * c_df['Sil_norm']  +
                        0.2 * c_df['Entropy_norm']).round(4)
c_df = c_df.sort_values('Composite', ascending=False)

print('=== Clustering intra-group comparison (C1, C2) ===')
print(c_df[['F1_macro','Silhouette','Entropy','Composite']].to_string())
BEST_CLUSTERING = c_df.index[0]
print(f'\nBest clustering variant: {BEST_CLUSTERING}')

# ── Supervised propagation — reference only ───────────────────────────────────
print('\n--- Supervised propagation (reference, not included in comparison) ---')
for name, col in [('C3 (KNN)', 'C3'), ('C4 (Random Forest)', 'C4')]:
    pred_codes = labelled[col].map(SEG_MAP).fillna(-1).astype(int).values
    sil = safe_silhouette(X_lab_s, pred_codes)
    f1  = f1_score(labelled['Market_segment'], labelled[col],
                   average='macro', zero_division=0)
    ent = distribution_entropy(unlabelled[col])
    print(f'  {name}: F1={f1:.4f}, Silhouette={sil:.4f}, Entropy={ent:.4f}')


In [ ]:
# ── PCA projection — C1 and C2 (clustering methods) ─────────────────────────
pca = PCA(n_components=2, random_state=RANDOM_STATE)
pca_coords = pca.fit_transform(X_lab_s)
var_exp    = pca.explained_variance_ratio_ * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, name, col in zip(axes,
                          ['C1 (KMeans)', 'C2 (DBSCAN)'],
                          ['C1', 'C2']):
    for seg in SEG_ORDER:
        mask = labelled[col] == seg
        ax.scatter(pca_coords[mask, 0], pca_coords[mask, 1],
                   c=[SEG_PALETTE[seg]], label=seg, alpha=0.55, s=18)
    f1  = cluster_results[name]['F1_macro']
    cmp = c_df.loc[name, 'Composite']
    ax.set_title(f'{name}  |  F1={f1:.3f}  |  Composite={cmp:.3f}',
                 fontsize=9, fontweight='bold')
    ax.set_xlabel(f'PC1 ({var_exp[0]:.1f}%)', fontsize=8)
    ax.set_ylabel(f'PC2 ({var_exp[1]:.1f}%)', fontsize=8)
    ax.tick_params(labelsize=7.5)

axes[0].legend(title='Segment', fontsize=7, title_fontsize=7.5, frameon=False)
fig.suptitle('Figure 8. PCA projection — clustering methods (labelled subset)',
             fontsize=10, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig8_pca_clustering.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Figure: all method distributions on unlabelled subset ────────────────────
all_methods = ['Manual_Label', 'H1', 'H2', 'H3', 'C1', 'C2', 'C3', 'C4']
method_labels = ['Manual', 'H1', 'H2', 'H3', 'C1\n(KMeans)', 'C2\n(DBSCAN)',
                 'C3\n(KNN)', 'C4\n(RF)']

fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharey=True)
axes = axes.flatten()

for ax, col, lbl in zip(axes, all_methods, method_labels):
    counts = unlabelled[col].value_counts().reindex(SEG_ORDER, fill_value=0)
    bars = ax.bar(SEG_ORDER, counts.values,
                  color=[SEG_PALETTE[s] for s in SEG_ORDER], edgecolor='white')
    for bar, v in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.2, str(v),
                ha='center', va='bottom', fontsize=7.5)
    ax.set_title(lbl, fontsize=9, fontweight='bold')
    ax.set_xlabel('Segment', fontsize=7.5)
    ax.tick_params(axis='x', rotation=30, labelsize=7)
    ax.tick_params(axis='y', labelsize=7)
axes[0].set_ylabel('Count')
axes[4].set_ylabel('Count')

fig.suptitle('Figure S3. Label distributions — all methods (unlabelled, n=100)',
             fontsize=10, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_all_distributions.png', bbox_inches='tight')
plt.show()


---
## Step 5 — Evaluation

### 5.1 Composite Score

The composite score used throughout is:

$$\text{Composite} = 0.6 \cdot F1_{macro} + 0.2 \cdot \text{Silhouette}_{norm} + 0.2 \cdot \text{Entropy}_{norm}$$

where $F1_{macro}$ is evaluated against ground truth on the labelled subset, Silhouette is normalised by the maximum value in the group, and Entropy is normalised by $\log(5)$ (maximum entropy for 5 classes). F1 is weighted at 60% as the primary quality criterion; geometric structure and distributional balance each contribute 20%.


In [ ]:
# ── Inter-group comparison: Manual vs Best Heuristic vs Best Clustering ───────
# Only unsupervised clustering methods (C1, C2) are eligible for the clustering
# candidate. The best is selected from c_df computed above.

best_cluster_col = BEST_CLUSTERING.split(' ')[0]   # 'C1' or 'C2'
best_cluster_clust_col = best_cluster_col + '_cluster'

candidates = {
    'Manual'                              : 'Manual_Label',
    f'Best Heuristic ({BEST_HEURISTIC})'  : BEST_HEURISTIC,
    f'Best Clustering ({BEST_CLUSTERING})': best_cluster_col,
}

inter_results = {}
for label, col in candidates.items():
    f1  = f1_score(labelled['Market_segment'], labelled[col],
                   average='macro', zero_division=0)
    ent = distribution_entropy(unlabelled[col])
    if col == 'C1':
        sil = safe_silhouette(X_lab_s, labelled['C1_cluster'].values)
    elif col == 'C2':
        sil = safe_silhouette(X_lab_s, labelled['C2_cluster'].values)
    else:
        pred_codes = labelled[col].map(SEG_MAP).fillna(-1).astype(int).values
        sil = safe_silhouette(X_lab_s, pred_codes)
    inter_results[label] = {'F1_macro': round(f1, 4),
                             'Silhouette': round(sil, 4),
                             'Entropy': round(ent, 4)}

inter_df = pd.DataFrame(inter_results).T
sil_max2 = inter_df['Silhouette'].max() + 1e-9
inter_df['Sil_norm']     = inter_df['Silhouette'] / sil_max2
inter_df['Entropy_norm'] = inter_df['Entropy']    / math.log(5)
inter_df['Composite']    = (0.6 * inter_df['F1_macro'] +
                            0.2 * inter_df['Sil_norm']  +
                            0.2 * inter_df['Entropy_norm']).round(4)
inter_df = inter_df.sort_values('Composite', ascending=False)

print('=== Inter-group comparison ===')
print(inter_df[['F1_macro','Silhouette','Entropy','Composite']].to_string())
GOLD_STANDARD = inter_df['Composite'].idxmax()
print(f'\nGold Standard candidate: {GOLD_STANDARD}')


In [ ]:
# ── Figure: inter-group composite score ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
methods_inter = list(inter_df.index)
composites    = inter_df['Composite'].values
colors_inter  = ['#2196F3', '#4CAF50', '#FF5722'][:len(methods_inter)]

bars = ax.bar(methods_inter, composites, color=colors_inter, edgecolor='white', width=0.4)
for bar, v in zip(bars, composites):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.002, f'{v:.4f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylim(0, max(composites) * 1.2)
ax.set_title('Figure 9. Inter-group composite score comparison',
             fontsize=10, fontweight='bold')
ax.set_ylabel('Composite score')
ax.tick_params(axis='x', labelsize=9)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig9_intergroup_composite.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Pairwise label agreement heatmap (all methods, unlabelled) ────────────────
all_cols = ['Manual_Label', 'H1', 'H2', 'H3', 'C1', 'C2', 'C3', 'C4']
all_lbls = ['Manual', 'H1', 'H2', 'H3', 'C1', 'C2', 'C3', 'C4']

agree_mat = pd.DataFrame(index=all_lbls, columns=all_lbls, dtype=float)
for i, c1 in enumerate(all_cols):
    for j, c2 in enumerate(all_cols):
        agree_mat.iloc[i, j] = (unlabelled[c1] == unlabelled[c2]).mean()

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(agree_mat.astype(float), annot=True, fmt='.2f',
            cmap='YlGn', vmin=0, vmax=1, ax=ax,
            linewidths=0.4, linecolor='white',
            annot_kws={'size': 9})
ax.set_title('Figure S4. Pairwise label agreement — all methods (unlabelled subset)',
             fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_agreement_heatmap.png', bbox_inches='tight')
plt.show()
print(agree_mat.to_string())


### 5.2 Candidate Datasets

Three candidate datasets are produced — one per labelling family — using the best variant identified by the composite score in each group. Each dataset contains the full 1,155 records with `Market_segment` fully populated.


In [ ]:
# ── Save candidate datasets ──────────────────────────────────────────────────
def build_candidate(label_col):
    lab_out   = labelled[list(df.columns)].copy()
    unlab_out = unlabelled[list(df.columns)].copy()
    unlab_out['Market_segment'] = unlabelled[label_col].values
    return pd.concat([lab_out, unlab_out], ignore_index=True)

manual_csv    = DATASETS_DIR / 'candidate_manual.csv'
heuristic_csv = DATASETS_DIR / f'candidate_heuristic_{BEST_HEURISTIC}.csv'
cluster_csv   = DATASETS_DIR / f'candidate_clustering_{best_cluster_col}.csv'

build_candidate('Manual_Label').to_csv(manual_csv,    index=False)
build_candidate(BEST_HEURISTIC).to_csv(heuristic_csv, index=False)
build_candidate(best_cluster_col).to_csv(cluster_csv,  index=False)

print('Candidate datasets saved:')
print(f'  {manual_csv}')
print(f'  {heuristic_csv}')
print(f'  {cluster_csv}')


### 5.3 Takeaways

- **Gold standard:** manual labelling, guided by the EDA, obtains the highest composite score.
- **Best automatic option:** H3 (Spearman-weighted score) — the most scalable alternative.
- **Without any labels:** KMeans describes the structure of the data better than DBSCAN, but houses form a continuum rather than well-separated segments.


### 5.4 Supervised Propagation Methods — Reference Results

KNN and Random Forest are not included in the main comparison as they are supervised methods, not clustering. Their results are shown here for reference, as a scalable extension of the pipeline.


In [ ]:
# ── Supervised propagation reference table ───────────────────────────────────
ref_rows = {}
for name, col in [('C3 (KNN)', 'C3'), ('C4 (Random Forest)', 'C4')]:
    pred_codes = labelled[col].map(SEG_MAP).fillna(-1).astype(int).values
    sil = safe_silhouette(X_lab_s, pred_codes)
    f1  = f1_score(labelled['Market_segment'], labelled[col],
                   average='macro', zero_division=0)
    ent = distribution_entropy(unlabelled[col])
    ref_rows[name] = {'F1_macro': round(f1,4),
                      'Silhouette': round(sil,4),
                      'Entropy': round(ent,4)}

ref_df = pd.DataFrame(ref_rows).T
print('=== Supervised propagation — reference ===')
print(ref_df.to_string())
print('\n(Not included in Gold Standard selection.)')


In [ ]:
# ── Figure: inter-group composite score (Manual / Heuristic / Clustering) ────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

method_names = list(inter_df.index)
composites   = inter_df['Composite'].values
colors_inter = ['#2196F3', '#4CAF50', '#FF9800']

# Left: composite bar
bars = axes[0].bar(method_names, composites,
                   color=colors_inter, edgecolor='white', width=0.45)
for bar, v in zip(bars, composites):
    axes[0].text(bar.get_x() + bar.get_width()/2, v + 0.005,
                 f'{v:.4f}', ha='center', va='bottom',
                 fontsize=10, fontweight='bold')
axes[0].set_ylim(0, max(composites) * 1.25)
axes[0].set_title('Composite score\n(Manual / Heuristic / Clustering)',
                  fontsize=9, fontweight='bold')
axes[0].set_ylabel('Composite score')
axes[0].tick_params(axis='x', labelsize=8, rotation=10)

# Right: metric breakdown
x     = np.arange(len(method_names))
width = 0.22
f1s   = inter_df['F1_macro'].values
sils  = inter_df['Sil_norm'].values
ents  = inter_df['Entropy_norm'].values

axes[1].bar(x - width, f1s,  width, label='F1-macro',         color='#1976D2', alpha=0.85)
axes[1].bar(x,         sils, width, label='Silhouette (norm)', color='#F57C00', alpha=0.85)
axes[1].bar(x + width, ents, width, label='Entropy (norm)',    color='#388E3C', alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels(method_names, fontsize=8, rotation=10)
axes[1].set_title('Metric breakdown\n(F1 / Silhouette / Entropy)',
                  fontsize=9, fontweight='bold')
axes[1].set_ylabel('Normalised score')
axes[1].legend(fontsize=8, frameon=False)
axes[1].set_ylim(0, 1.15)

fig.suptitle('Figure 9. Inter-group comparison — composite score and metric breakdown',
             fontsize=10, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig9_intergroup_comparison.png', bbox_inches='tight')
plt.show()
